# Interruptible MLE agent: one Kaggle session

Runs one session of `configs/kaggle_sessions.yaml`: one condition and seed on insults and pizza, 4 hours each,
one after the other (about 8.5 hours). Use **Save Version → Save & Run All (Commit)**: it runs in the
background with the browser closed, and everything in `/kaggle/working` is kept as the version's output.
See `docs/cloud-runs.md`.

**Before the first run** (once per notebook, `imle-slot-1` to `imle-slot-5`)
- *Settings*: Accelerator **None** (CPU), Internet **On** (needs a phone-verified account).
- *Add-ons → Secrets*: `GITHUB_TOKEN` (fine-grained, read-only *Contents* on this repo) and `OPENROUTER_API_KEY`.
- *Add Input*: the private dataset `imle-kaggle-data`, with `prepared/public` and `prepared/private` of both tasks.

**Each round**: set `SESSION`, then Save & Run All. Collect the outputs with `scripts/kaggle_collect.py` before
the next round, since only a notebook's latest version can be downloaded.

**If a session was cut off**: attach this notebook's own output as an input (*Add Input → Your Work → Notebooks*)
and run the same `SESSION` again. Each run continues from its last checkpoint there.

`TEST = True` runs both tasks for 15 minutes under `test_` names, to check the setup. Docker is not available,
so scripts run with `--sandbox subprocess` (no isolation, network access); the report states this.

In [ ]:
# Settings: the only cell to edit.
SESSION = 1               # 1 to 21, see configs/kaggle_sessions.yaml
TEST = False              # True: both tasks for 15 minutes, named test_..., to check the setup
BRANCH = "main"
PROVIDER = "openrouter"

In [ ]:
import os
import subprocess
import sys

REPO = "github.com/nabilrakaiza/interruptible-mle-agent.git"


def git(*args, token=""):
    """Run git without ever printing the token, even when it fails."""
    result = subprocess.run(["git", *args], capture_output=True, text=True, check=False)
    if result.returncode != 0:
        message = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError("git failed: " + message)


def get_code(dest, token):
    """Clone the private repo (or update it), then remove the token from .git/config."""
    url = f"https://x-access-token:{token}@{REPO}"
    if not os.path.exists(dest):
        git("clone", "-q", "--branch", BRANCH, url, dest, token=token)
    else:
        git("-C", dest, "fetch", "-q", url, BRANCH, token=token)
        git("-C", dest, "checkout", "-q", "FETCH_HEAD", token=token)
    git("-C", dest, "remote", "set-url", "origin", f"https://{REPO}")


def sh(*cmd, cwd=None):
    """Run a command and stop the notebook if it fails."""
    subprocess.run(list(cmd), cwd=cwd, check=True)


from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["OPENROUTER_API_KEY"] = secrets.get_secret("OPENROUTER_API_KEY")

CODE = "/tmp/imle"  # outside /kaggle/working, so the code is not saved as output
get_code(CODE, secrets.get_secret("GITHUB_TOKEN"))
COMMIT = subprocess.run(["git", "-C", CODE, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

# The sandbox image's Python and packages (Dockerfile, docker-requirements.txt), so the scripts the agent writes
# run with the versions its prompt lists, as on the Mac. Kaggle's own Python has other versions.
VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"
sh(sys.executable, "-m", "pip", "install", "-q", "uv")
sh(sys.executable, "-m", "uv", "venv", "-q", "--python", "3.13.6", VENV)
sh(sys.executable, "-m", "uv", "pip", "install", "-q", "--python", PY, "-r", "docker-requirements.txt", cwd=CODE)
sh(sys.executable, "-m", "uv", "pip", "install", "-q", "--python", PY, "torch==2.14.1",
   "--index-url", "https://download.pytorch.org/whl/cpu")
sh(sys.executable, "-m", "uv", "pip", "install", "-q", "--python", PY, "-e", ".[analysis]", cwd=CODE)
sh(PY, "-c", "import sys, numpy, pandas, scipy, sklearn, PIL, torch; "
   "print('python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__, "
   "'| scipy', scipy.__version__, '| scikit-learn', sklearn.__version__, '| pillow', PIL.__version__, "
   "'| torch', torch.__version__)")
print("code at", COMMIT)

In [ ]:
import glob
from pathlib import Path

import yaml

PLAN = yaml.safe_load(Path(CODE, "configs/kaggle_sessions.yaml").read_text())

# The evaluator reads $MLEBENCH_DATA_DIR/<task>/prepared/{public,private}. Find both wherever Kaggle mounted the
# dataset, and link them into that layout.
DATA_DIR = Path("/tmp/mlebench-data")
for task in PLAN["tasks"]:
    for part in ["public", "private"]:
        found = sorted(glob.glob(f"/kaggle/input/**/{task}/prepared/{part}", recursive=True))
        if not found:
            raise FileNotFoundError(f"no {task}/prepared/{part} in the attached data: add it to imle-kaggle-data")
        link = DATA_DIR / task / "prepared" / part
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists():
            link.symlink_to(found[0])
        print(link, "->", found[0])
os.environ["MLEBENCH_DATA_DIR"] = str(DATA_DIR)

In [ ]:
import json

RUNS_ROOT = Path("/kaggle/working/runs")
session = PLAN["sessions"][SESSION]
config = f"configs/conditions/{session['config']}"
condition = yaml.safe_load(Path(CODE, config).read_text())["condition"]
budget = 0.25 if TEST else PLAN["budget_hours"]

results = {}
for task in PLAN["tasks"]:
    name = f"{'test_' if TEST else ''}{condition}_{task}_seed{session['seed']}"
    cmd = [PY, "scripts/cloud_run.py", "--config", config, "--task", task, "--seed", str(session["seed"]),
           "--budget-hours", str(budget), "--sandbox", "subprocess", "--evaluator", "a1", "--provider", PROVIDER,
           "--runs-root", str(RUNS_ROOT), "--name", name]
    # A cut-off session: this notebook's earlier output, attached as input, holds the run to continue.
    earlier = sorted(glob.glob(f"/kaggle/input/**/runs/{name}", recursive=True))
    if earlier:
        cmd += ["--restore-from", str(Path(earlier[0]).parent)]
    print(" ".join(cmd), flush=True)
    # Stream the run's output into this cell as it happens.
    process = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    results[name] = process.wait()
    print(name, "exit code", results[name], flush=True)

# What scripts/kaggle_collect.py checks: which session this output is, and with which code.
Path("/kaggle/working/session.json").write_text(json.dumps(
    {"session": SESSION, "test": TEST, "commit": COMMIT, "budget_hours": budget, "exit_codes": results}, indent=2))
print(results)

## Look at the result

The trajectory figure and timeline are written into each run folder, so they are part of the output, and the
runs are also packed into `runs.tar.gz` for downloading.

In [ ]:
for run in sorted(RUNS_ROOT.glob("*")):
    subprocess.run([PY, "scripts/trajectory.py", str(run)], cwd=CODE, check=False, capture_output=True)
    print(run, "->", run / "trajectory")

# One archive of the runs, so scripts/kaggle_collect.py downloads one file instead of thousands.
sh("tar", "-czf", "/kaggle/working/runs.tar.gz", "-C", "/kaggle/working", "runs")
print("runs.tar.gz", round(os.path.getsize("/kaggle/working/runs.tar.gz") / 1e6, 1), "MB")